# coreX V2 — OmniAnomaly Training
**Feature Engineered RobotArm Anomaly Detection**

Settings Required:
- GPU T4 x2

In [1]:
%%bash
# === CELL 1: Copy Project ===
cd /kaggle/working
rm -rf project

SRC=$(find /kaggle/input -type d -name "Omnia_Anomaly_Detection_coreX-main" | head -1)

if [ -n "$SRC" ]; then
    cp -r "$SRC" project
    echo "✅ Project copied from: $SRC"
    ls project/
else
    echo "❌ Could not find project folder. Listing input:"
    find /kaggle/input -maxdepth 3 -type d
fi

✅ Project copied from: /kaggle/input/datasets/mahmoudsalaheldeen/corex-v2/Omnia_Anomaly_Detection_coreX-main
anomaly_factory.py
check_gpu.py
Colab_VSCode_Guide.md
data
data_preprocess.py
explore_data.py
feature_engineering.py
fix_data.py
inspect_pkl.py
main.py
omni_anomaly
plot_results.py
quick_results.py
README.md
requirements.txt
RobotArm_final_report.png
run_all.bat
run_preprocess.bat
SSH_Success_Workflow.md
USAGE_GUIDE.md


In [3]:
%%bash
# === CELL 2: Fixed Environment Setup ===
set -e

# 1. Install Miniconda if not present
if ! command -v /opt/conda/bin/conda &> /dev/null; then
    echo "=== Installing Miniconda ==="
    wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /tmp/miniconda.sh
    bash /tmp/miniconda.sh -b -p /opt/conda -f 2>/dev/null
fi
export PATH=/opt/conda/bin:$PATH

# 2. Accept TOS & Configure
conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main 2>/dev/null || true
conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r 2>/dev/null || true
conda config --set always_yes true

# 3. Create corex_env (Python 3.6)
echo "=== Creating corex_env ==="
conda env remove -n corex_env 2>/dev/null || true
conda create -n corex_env python=3.6 -y -q

# 4. Install TF 1.15 + CUDA + Data Stack
echo "=== Installing TF 1.15 + CUDA (This takes a few minutes) ==="
conda install -n corex_env -c defaults tensorflow-gpu=1.15 cudatoolkit=10.0 cudnn=7 numpy=1.16 scipy pandas scikit-learn matplotlib -y -q

# 5. Install specialized AI dependencies
PIP=/opt/conda/envs/corex_env/bin/pip
echo "=== Installing pip dependencies ==="
$PIP install -q \
    git+https://github.com/haowen-xu/tfsnippet.git@v0.2.0-alpha1 \
    git+https://github.com/thu-ml/zhusuan.git \
    seaborn \
    imageio

echo ""
echo "=== Verifying installs (Fixing Matplotlib Backend) ==="
# We force MPLBACKEND=Agg to prevent 'matplotlib_inline' errors in headless Kaggle
MPLBACKEND=Agg /opt/conda/envs/corex_env/bin/python -c "
import tfsnippet; print('tfsnippet OK')
import zhusuan; print('zhusuan OK:', zhusuan.__version__)
import seaborn; print('seaborn OK')
import imageio; print('imageio OK')
"

echo ""
echo "✅✅✅ ENVIRONMENT FULLY READY ✅✅✅"


accepted Terms of Service for https://repo.anaconda.com/pkgs/main
accepted Terms of Service for https://repo.anaconda.com/pkgs/r
=== Creating corex_env ===
Jupyter detected...
2 channel Terms of Service accepted

Remove all packages in environment /opt/conda/envs/corex_env:


## Package Plan ##

  environment location: /opt/conda/envs/corex_env


The following packages will be REMOVED:

  _libgcc_mutex-0.1-main
  _openmp_mutex-5.1-1_gnu
  _tflow_select-2.1.0-gpu
  absl-py-0.15.0-pyhd3eb1b0_0
  astor-0.8.1-py36h06a4308_0
  blas-1.0-openblas
  c-ares-1.34.6-hd44998d_0
  ca-certificates-2025.12.2-h06a4308_0
  certifi-2021.5.30-py36h06a4308_0
  coverage-5.5-py36h27cfd23_2
  cudatoolkit-10.0.130-0
  cudnn-7.6.5-cuda10.0_0
  cupti-10.0.130-0
  cycler-0.11.0-pyhd3eb1b0_0
  cython-0.29.24-py36h295c915_0
  dbus-1.16.2-h5bd4931_0
  expat-2.7.4-h7354ed3_0
  fontconfig-2.14.1-h52c9d5c_1
  freetype-2.14.1-hf5b9546_0
  gast-0.2.2-py36_0
  giflib-5.2.2-h5eee18b_0
  glib-2.69.1-h4ff587b_1
  google-pas

In [4]:
%%bash
# === CELL 3: Verify GPU ===
export PATH=/opt/conda/envs/corex_env/bin:$PATH
/opt/conda/envs/corex_env/bin/python -c "
import tensorflow as tf
print('TF Version:', tf.__version__)
print('GPU Available:', tf.test.is_gpu_available())
print('GPU Name:', tf.test.gpu_device_name())
"

TF Version: 1.15.0
GPU Available: True
GPU Name: /device:GPU:0


In [15]:
# === CELL 4: Patch main.py ===
# ⚡ CHANGE max_epoch HERE: 2 for test, 50 for production
EPOCHS = 50

import os, re
os.chdir('/kaggle/working/project')

with open('main.py', 'r') as f:
    content = f.read()

content = re.sub(r'max_epoch\s*=\s*\d+', f'max_epoch = {EPOCHS}', content)
content = re.sub(r"'max_epoch':\s*\d+", f"'max_epoch': {EPOCHS}", content)
content = re.sub(r'batch_size\s*=\s*\d+', 'batch_size = 25', content)
content = re.sub(r"restore_dir\s*=\s*'[^']*'", 'restore_dir = None', content)

with open('main.py', 'w') as f:
    f.write(content)

print(f'✅ main.py patched: {EPOCHS} epochs, batch_size=25, fresh training')

✅ main.py patched: 50 epochs, batch_size=25, fresh training


In [6]:
%%bash
# === CELL 5: Preprocess Data (V2 Feature Engineering) ===
cd /kaggle/working/project
MPLBACKEND=Agg /opt/conda/envs/corex_env/bin/python -u data_preprocess.py

CoreX Preprocessing System Starting for: RobotArm
--- [Step 1] Loading Raw Data...

--- [Step 2] Unpacking Vectors (Making it AI-Ready)...
[OK] Unpacked: target_q (6 elements)
[OK] Unpacked: target_qd (6 elements)
[OK] Unpacked: target_qdd (6 elements)
[OK] Unpacked: target_current (6 elements)
[OK] Unpacked: target_moment (6 elements)
[OK] Unpacked: actual_q (6 elements)
[OK] Unpacked: actual_qd (6 elements)
[OK] Unpacked: actual_current (6 elements)
[OK] Unpacked: joint_temperatures (6 elements)
[OK] Unpacked: target_TCP_pose (6 elements)
[OK] Unpacked: actual_TCP_pose (6 elements)
[OK] Unpacked file saved at: data/RobotArm/ready_for_ai.csv

--- [Step 3] Running Data Inventory Audit...

[coreX Data Audit] Full Detailed Inventory Starting...
--- Dataset Shape: 33486 Rows, 68 Columns

--- [Full Column Data Types]:
timestamp            float64
target_q_0           float64
target_q_1           float64
target_q_2           float64
target_q_3           float64
                      ...   


In [12]:
%%bash
/opt/conda/envs/corex_env/bin/pip install \
    tensorflow-probability==0.8.0 \
    tqdm==4.28.1 \
    fs==2.3.0 \
    click==7.0 \
    PyYAML==5.4.1 \
    xlrd==2.0.2 \
    openpyxl==3.1.3



  Attempting uninstall: tqdm
    Found existing installation: tqdm 4.64.1
    Uninstalling tqdm-4.64.1:
      Successfully uninstalled tqdm-4.64.1
  Attempting uninstall: PyYAML
    Found existing installation: PyYAML 6.0.1
    Uninstalling PyYAML-6.0.1:
      Successfully uninstalled PyYAML-6.0.1
  Attempting uninstall: fs
    Found existing installation: fs 2.4.16
    Uninstalling fs-2.4.16:
      Successfully uninstalled fs-2.4.16
  Attempting uninstall: click
    Found existing installation: click 8.0.4
    Uninstalling click-8.0.4:
      Successfully uninstalled click-8.0.4


In [1]:
%%bash
# === CELL 6: TRAIN ===
cd /kaggle/working/project
MPLBACKEND=Agg PYTHONPATH=. /opt/conda/envs/corex_env/bin/python -u main.py

bash: line 2: cd: /kaggle/working/project: No such file or directory
bash: line 3: /opt/conda/envs/corex_env/bin/python: No such file or directory


CalledProcessError: Command 'b'# === CELL 6: TRAIN ===\ncd /kaggle/working/project\nMPLBACKEND=Agg PYTHONPATH=. /opt/conda/envs/corex_env/bin/python -u main.py\n'' returned non-zero exit status 127.

In [ ]:
# === CELL 7: Export Results ===
import zipfile, os, glob

os.chdir('/kaggle/working/project')

def make_zip(name, patterns):
    with zipfile.ZipFile(f'/kaggle/working/{name}', 'w', zipfile.ZIP_DEFLATED) as z:
        for p in patterns:
            for f in glob.glob(p, recursive=True):
                z.write(f)
    print(f'✅ {name} created')

make_zip('coreX_v2_results.zip', ['results/**/*'])
make_zip('coreX_v2_model.zip', ['model_coreX_v2_optimized/**/*'])

print('\n🎉 Download your results from the Output tab!')